## BU

Bottom-up **order reconciliation** using the cleaned simulation pipeline.

In [1]:
path1 = 'E:/yjz/Extension for hts/JayCode/Models/'

import warnings
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp

from inventory_pipeline import save_pickle, run_fixed_loop

In [2]:
test = pd.read_pickle(f"{path1}721future_28.pkl").reset_index(drop=True)
ts_id = test[['unique_id', 'ds']].reset_index(drop=True)
truth = test['y'].reset_index(drop=True)

S = pd.read_pickle(f"{path1}721S_df.pkl")
tags = pd.read_pickle(f"{path1}tags.bin")

lead_time = 1
MODEL_NAMES = ['lgb_base', 'lgb_bu', 'lgb_td', 'lgb_mint',
               'ets_base', 'ets_bu', 'ets_td', 'ets_mint']
on = ['ot_90', 'ot_95', 'ot_99']

lgbs = pd.read_pickle(f"lgbInvtSim_L{lead_time}.pkl").reset_index(drop=True)
etss = pd.read_pickle(f"etsInvtSim_L{lead_time}.pkl").reset_index(drop=True)
base_all = pd.concat([lgbs, etss], ignore_index=True)

bu = HierarchicalReconciliation(reconcilers=[BottomUp()])

In [3]:
base_all.iloc[:28,:].to_csv('base_sample.csv')

In [4]:
lobu = []
for name in tqdm(MODEL_NAMES):
    sub_orders = base_all.loc[base_all['name'] == name, on].reset_index(drop=True)
    y_hat = pd.concat([ts_id, sub_orders], axis=1)
    rec = bu.reconcile(Y_hat_df=y_hat, S=S, tags=tags).iloc[:,-3:].copy()
    rec.insert(0, 'name', name)
    lobu.append(rec.reset_index(drop=True))

OrderBU = pd.concat(lobu, ignore_index=True)
save_pickle(OrderBU, f"OrderBU_L{lead_time}.pkl")
OrderBU.head()

100%|███████████████████████████████████████████████████████████████████████████████████| 8/8 [20:48<00:00, 156.05s/it]


,name,ot_90/BottomUp,ot_95/BottomUp,ot_99/BottomUp
0,lgb_base,19.079405,23.543574,31.917614
1,lgb_base,5.387234,5.387234,5.387234
2,lgb_base,7.938419,7.938419,7.938419
3,lgb_base,1.017425,1.017425,1.017425
4,lgb_base,12.051126,12.051126,12.051126


In [5]:
['name']+on

['name', 'ot_90', 'ot_95', 'ot_99']

In [6]:
OrderBU.columns = ['name']+on
OrderBU

,name,ot_90,ot_95,ot_99
0,lgb_base,19.079405,23.543574,31.917614
1,lgb_base,5.387234,5.387234,5.387234
2,lgb_base,7.938419,7.938419,7.938419
3,lgb_base,1.017425,1.017425,1.017425
4,lgb_base,12.051126,12.051126,12.051126
...,...,...,...,...
9561659,ets_mint,0.000000,0.000000,0.000000
9561660,ets_mint,0.021772,0.021772,0.021772
9561661,ets_mint,0.025888,0.025888,0.025888
9561662,ets_mint,0.000000,0.000000,0.000000


In [7]:
Invt_df = []
for name in MODEL_NAMES:
    base_ref = base_all.loc[base_all['name'] == name].reset_index(drop=True)
    order_ref = OrderBU.loc[OrderBU['name'] == name].reset_index(drop=True)

    fixed_orders = pd.concat(
        [
            base_ref[['forecasts', 'sst_90', 'sst_95', 'sst_99']].reset_index(drop=True),
            order_ref[['ot_90', 'ot_95', 'ot_99']].reset_index(drop=True),
        ],
        axis=1,
    )

    df = run_fixed_loop(
        truth=truth,
        forecast_source=base_ref['forecasts'].reset_index(drop=True),
        fixed_orders=fixed_orders,
        NAME=name,
        L_=lead_time,
    )
    Invt_df.append(df)

BUOrder = pd.concat(Invt_df, ignore_index=True)
save_pickle(BUOrder, f"BUOrder_L{lead_time}.pkl")
BUOrder.head()

100%|███████████████████████████████████████████████████████████████████████████| 42686/42686 [01:19<00:00, 536.00it/s]


,name,true_demand,forecasts,sst_90,arrival_90,ot_90,wip_90,ip_90,net_90,backlog_90,...,cb_95,sst_99,arrival_99,ot_99,wip_99,ip_99,net_99,backlog_99,ch_99,cb_99
0,lgb_base,4.0,5.689794,5.648312,0.000000,19.079405,19.079405,20.769199,1.689794,0.0,...,0.0,10.253148,0.000000,31.917614,31.917614,33.607408,1.689794,0.0,1.689794,0.0
1,lgb_base,5.0,4.679130,5.648312,19.079405,5.387234,5.387234,21.156433,15.769199,0.0,...,0.0,10.253148,31.917614,5.387234,5.387234,33.994642,28.607408,0.0,28.607408,0.0
2,lgb_base,7.0,4.798928,5.648312,5.387234,7.938419,7.938419,22.094851,14.156433,0.0,...,0.0,10.253148,5.387234,7.938419,7.938419,34.933061,26.994642,0.0,26.994642,0.0
3,lgb_base,1.0,5.980217,5.648312,7.938419,1.017425,1.017425,22.112276,21.094851,0.0,...,0.0,10.253148,7.938419,1.017425,1.017425,34.950486,33.933061,0.0,33.933061,0.0
4,lgb_base,9.0,5.048564,5.648312,1.017425,12.051126,12.051126,25.163402,13.112276,0.0,...,0.0,10.253148,1.017425,12.051126,12.051126,38.001612,25.950486,0.0,25.950486,0.0


In [8]:
import pandas as pd
lead_time = 1
a = pd.read_pickle(f"BUOrder_L{lead_time}.pkl")

In [9]:
a['name'].unique()

array(['lgb_base', 'lgb_bu', 'lgb_td', 'lgb_mint', 'ets_base', 'ets_bu',
       'ets_td', 'ets_mint'], dtype=object)

In [10]:
a.iloc[:28,:12]

,name,true_demand,forecasts,sst_90,arrival_90,ot_90,wip_90,ip_90,net_90,backlog_90,ch_90,cb_90
0,lgb_base,4.0,5.689794,5.648312,0.000000,19.079405,19.079405,20.769199,1.689794,0.0,1.689794,0.0
1,lgb_base,5.0,4.679130,5.648312,19.079405,5.387234,5.387234,21.156433,15.769199,0.0,15.769199,0.0
2,lgb_base,7.0,4.798928,5.648312,5.387234,7.938419,7.938419,22.094851,14.156433,0.0,14.156433,0.0
3,lgb_base,1.0,5.980217,5.648312,7.938419,1.017425,1.017425,22.112276,21.094851,0.0,21.094851,0.0
4,lgb_base,9.0,5.048564,5.648312,1.017425,12.051126,12.051126,25.163402,13.112276,0.0,13.112276,0.0
5,lgb_base,3.0,8.247916,5.648312,12.051126,2.722735,2.722735,24.886137,22.163402,0.0,22.163402,0.0
6,lgb_base,9.0,6.159383,5.648312,2.722735,8.339869,8.339869,24.226006,15.886137,0.0,15.886137,0.0
7,lgb_base,1.0,6.263591,5.648312,8.339869,1.126628,1.126628,24.352634,23.226006,0.0,23.226006,0.0
8,lgb_base,10.0,4.828214,5.648312,1.126628,9.552660,9.552660,23.905294,14.352634,0.0,14.352634,0.0
9,lgb_base,3.0,6.094202,5.648312,9.552660,3.094780,3.094780,24.000074,20.905294,0.0,20.905294,0.0
